In [1]:
import numpy as np
import pandas as pd
 
from scipy import stats
 

In [2]:
np.random.seed(42)
 
n = 60
 
# A/B pavyzdys: vidutinė užsakymo suma (EUR) dviem grupėms
group_A = np.random.normal(loc=42, scale=8, size=n)  # kontrolė
group_B = np.random.normal(loc=45, scale=8, size=n)  # testas (šiek tiek didesnis vidurkis)
 
df_ab = pd.DataFrame({
    "group": ["A"] * n + ["B"] * n,
    "order_value": np.concatenate([group_A, group_B])
})
 
df_ab.head()
 

,group,order_value
0,A,45.973713
1,A,40.893886
2,A,47.181508
3,A,54.184239
4,A,40.126773


In [3]:
a = df_ab.loc[df_ab["group"] == "A", "order_value"].values
b = df_ab.loc[df_ab["group"] == "B", "order_value"].values
 
a_mean, b_mean = a.mean(), b.mean()
a_mean, b_mean
 

(np.float64(40.76276253543608), np.float64(44.97053258366812))

In [4]:
t_stat, p_value = stats.ttest_ind(a, b, equal_var=False)  # Welch t testas
t_stat, p_value

(np.float64(-3.110727229567604), np.float64(0.0023412701501776645))

In [5]:
diff = b_mean - a_mean
diff

np.float64(4.207770048232035)

In [6]:
def cohens_d(x, y):
    # Cohen d (naudojamas sujungtas SD; Welch atveju tai apytikslis rodiklis)
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    s_pooled = np.sqrt(((nx - 1) * sx**2 + (ny - 1) * sy**2) / (nx + ny - 2))
    return (y.mean() - x.mean()) / s_pooled
 
d = cohens_d(a, b)
d

np.float64(0.5679384912932427)

In [7]:
np.random.seed(7)
 
n_customers = 50
before = np.random.normal(loc=120, scale=25, size=n_customers)
# tarkime, po pokyčio vidutiniškai +8 EUR, bet su triukšmu
after = before + np.random.normal(loc=8, scale=15, size=n_customers)
 
t_stat_paired, p_value_paired = stats.ttest_rel(after, before)
t_stat_paired, p_value_paired

(np.float64(4.870447205269689), np.float64(1.2046137681231104e-05))

In [8]:
np.random.seed(10)
 
n = 400
customer_type = np.random.choice(["Member", "Normal"], size=n, p=[0.55, 0.45])
 
# sukuriama nedidelė priklausomybė: Members dažniau renkasi Ewallet
payment = []
for ct in customer_type:
    if ct == "Member":
        payment.append(np.random.choice(["Cash", "Card", "Ewallet"], p=[0.35, 0.30, 0.35]))
    else:
        payment.append(np.random.choice(["Cash", "Card", "Ewallet"], p=[0.45, 0.35, 0.20]))
 
df_pay = pd.DataFrame({"CustomerType": customer_type, "Payment": payment})
 
ct_table = pd.crosstab(df_pay["CustomerType"], df_pay["Payment"])
ct_table

Payment,Card,Cash,Ewallet
CustomerType,,,
Member,53,80,90
Normal,56,86,35


In [9]:
chi2, p, dof, expected = stats.chi2_contingency(ct_table)
chi2, p, dof

(np.float64(19.466885842485844), np.float64(5.92678877094638e-05), 2)

In [16]:
# statsmodels gali nebūti reikalingas, tačiau proporcijoms jis yra patogus
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest

In [17]:
np.random.seed(21)
 
nA, nB = 1200, 1200
convA = np.random.binomial(1, 0.075, size=nA)  # 7.5% konversija
convB = np.random.binomial(1, 0.090, size=nB)  # 9.0% konversija
 
successes = np.array([convA.sum(), convB.sum()])
trials = np.array([nA, nB])
 
z_stat, p_val = proportions_ztest(count=successes, nobs=trials, alternative="two-sided")
successes, trials, z_stat, p_val

(array([89, 93]),
 array([1200, 1200]),
 np.float64(-0.3084246973793868),
 np.float64(0.7577591915581285))